<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter9/Gradio_Interference_Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Understanding the Interface class

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In this example we will build an audio-to-audio fucntion that takes an audio file and simply reverses it.

We will use for the input the Audio component. When using the Audio component, we can specify whether we want the source of the audio to be a file that the user uploads or a microphone that the user records their voice with. In this case, we will set it to a "microphone". Just for fun, we'll add a label to our Audio that says "Speak"

In addition, we'd like to receive the audio as a numpy array so that we can easily 'reverse' it. So we'll set the "type" to be "numpy", which passes the input data as a tuple of (sample_rate, data) into our function.

We will also use the Audio output component which can automatically render a tuple with a sample rate and numpy array of data as a playable audio file. In this case, we do not need to do any customization, so we will use the string shortcut "audio".

In [3]:
import numpy as np
import gradio as gr

def reverse_audio(audio):
  sr, data = audio
  reversed_audio = (sr, np.flipud(data))
  return reversed_audio

mic = gr.Audio(sources=['microphone'], type='numpy', label='Speak')
gr.Interface(reverse_audio, mic, 'audio').launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff285c6947624623e4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Handling multiple inputs and outputs ##

Let's say we had a more complicated function, with multiple inputs and outputs. In the example below, we have a function that takes a dropdown index, a slider value, and number, and returns an audio sample of a musical tone.

The code snippet below shows how three input components line up with the three arguments of the generate_tone() function:

In [7]:
import numpy as np
import gradio as gr

notes = ["C", 'C#', 'D', 'D#', 'E', 'F', 'G', 'G#', 'A', 'A#', 'B']

def generate_tone(note, octave, duration):
  sr = 200000 # the sampling rate of 1 second of audio
  a4_freq, tones_from_a4 = 440, 12 * (octave - 4) + (note - 9)
  frequency = a4_freq * 2 ** (tones_from_a4 / 12) # Calculates the frequency in Hz
  duration = int(duration)
  audio = np.linspace(0, duration, duration * sr)
  audio = (20000 * np.sin(audio * (2 * np.pi * frequency))).astype(np.int16)
  return (sr, audio)

gr.Interface(
    generate_tone,
    [
        gr.Dropdown(notes, type='index'),
        gr.Slider(minimum=4, maximum=6, step=1),
        gr.Number(value=1, label='Duration in seconds')
    ],
    'audio',
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc768b5ff5b34e0f6d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Let's now build an interface that allows us to demo a speech-recognition model. To make it more interesting, we will accept either a mic input or an uploaded file.

As usual, we'll load our sr model using the pipeline() function from HuggingFace Transformers. Next, we'll implement a transcribe_audio() function that preprocesses the audio and returns the transcription. Finally, we'll wrap this function in an Interface with the Audio components for the inputs and just text for the output. Altogether, the code for this app is the following:

In [ ]:
from transformers import pipeline
import gradio as gr

model = pipeline('automatic-speech-recognition')

def transcribe_audio(audio):
  transcription = model(audio)['text']
  return transcription

gr.Interface(
    fn=transcribe_audio,
    inputs=gr.Audio(type='filepath'),
    outputs='text',
).launch()

[transformers] No model was supplied, defaulted to facebook/wav2vec2-base-960h and revision 22aad52.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

That is it, we can now use this interface to transcribe audio. Notice here that by passing in the optional parameter as True, we allow the user to either provide a microphone or an audio file (or neither, but that will return an error message)